# Módulo 21: Práctica de Prompt & Context Engineering con OpenAI API
**Diplomado en Inteligencia Artificial y Salud 2026**

Este Jupyter Notebook es el recurso práctico que complementa la **Presentación 21**. Sigue estrictamente la secuencia de las diapositivas para ejecutar ejemplos en vivo con la **API de OpenAI**:

1. **Configuración del Cliente OpenAI** (Soporte API Key + MOCK Fallback).
2. **Anatomía de Prompts & Delimitadores XML** *(Diapositivas 03-05)*.
3. **Zero-Shot vs. Few-Shot Prompting (In-Context Learning)** *(Diapositivas 06-07)*.
4. **System Prompts & Asignación de Roles** *(Diapositiva 08)*.
5. **Salidas Estructuradas con Pydantic (`Structured Outputs`)** *(Diapositiva 09)*.
6. **Control de Parámetros: Temperature & Top-P** *(Diapositiva 10)*.
7. **Chain-of-Thought (CoT) Reasoning** *(Diapositiva 11)*.
8. **Self-Consistency Prompting (Wang et al., 2022)** *(Diapositiva 12)*.
9. **Tree-of-Thoughts (ToT) Prompting (Yao et al., 2023)** *(Diapositiva 13)*.
10. **Patrón ReAct & Context Engineering (Scratchpad de Memoria)** *(Diapositivas 15, 21-23)*.

## 1. Configuración del Cliente OpenAI
Importamos las librerías necesarias e inicializamos el cliente de `openai`. Si no se detecta la variable de entorno `OPENAI_API_KEY`, el notebook ejecutará automáticamente respuestas MOCK para garantizar que todas las celdas corran limpiamente.

In [ ]:
import os
import json
from collections import Counter
from typing import List, Optional
from pydantic import BaseModel, Field

# Intentamos importar la librería oficial de OpenAI
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    HAS_OPENAI = False
    print("[WARN] La librería 'openai' no está instalada. Puedes instalarla ejecutando: !pip install openai pydantic")

# Verificación de API Key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY and HAS_OPENAI:
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("[OK] Cliente OpenAI configurado correctamente con tu API Key.")
else:
    client = None
    print("[INFO] OPENAI_API_KEY no detectada en el entorno. Los ejemplos incluirán respuestas MOCK para poder probar la lógica sin errores.")
    print("       (Para probar en vivo, define la variable: os.environ['OPENAI_API_KEY'] = 'sk-...')")


### Función Auxiliar para Llamadas a la API
Creamos `completar_prompt()` como envoltorio para interactuar con `gpt-4o-mini`, permitiendo configurar `temperature` y `top_p`.

In [ ]:
def completar_prompt(prompt: str, system_prompt: str = "Eres un asistente médico experto e investigador de IA en salud.", model: str = "gpt-4o-mini", temperature: Optional[float] = None, top_p: Optional[float] = None) -> str:
    """Envía un prompt a la API de OpenAI configurando temperature o top_p."""
    kwargs = {"model": model, "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]}
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p
        
    if client:
        try:
            response = client.chat.completions.create(**kwargs)
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"[ERROR] Error al invocar la API de OpenAI: {e}"
    else:
        params_str = f"temp={temperature}" if temperature is not None else f"top_p={top_p}"
        return f"[MOCK RESPONSE - {model} ({params_str})]\nRazonamiento generado exitosamente a partir de las instrucciones brindadas."


## 2. Anatomía de Prompts & Delimitadores XML (Diapositivas 03-05)
Uso de etiquetas XML (`<instructions>`, `<patient_data>`, `<output_format>`) para separar claramente los componentes del prompt y evitar confusiones en la atención del modelo.

In [ ]:
def construir_prompt_delimitado(instrucciones: str, datos_paciente: str) -> str:
    prompt_template = f"""<instructions>
{instrucciones}
</instructions>

<patient_data>
{datos_paciente}
</patient_data>

<output_format>
Responde exclusivamente en formato estructurado con: 1. Síntomas clave, 2. Hipótesis principal, 3. Exámenes requeridos.
</output_format>"""
    return prompt_template

inst = "Analiza la historia de urgencias y determina la conducta inicial."
datos = "Paciente femenino de 45 años consulta por dolor abdominal agudo en fosa ilíaca derecha, fiebre de 38.5C y signo de Blumberg positivo."

prompt_xml = construir_prompt_delimitado(inst, datos)
print("--- PROMPT ENSAMBLADO CON ETIQUETAS XML ---")
print(prompt_xml)
print("\n" + "="*70 + "\n")
res_xml = completar_prompt(prompt_xml, temperature=0.0)
print(res_xml)


## 3. Zero-Shot vs. Few-Shot Prompting (Diapositivas 06-07)
Comparamos la respuesta del LLM sin ejemplos previa (**Zero-Shot**) frente a suministrarle ejemplos in-context (**Few-Shot**) para estandarizar la clasificación de severidad de notas médicas.

In [ ]:
# Zero-Shot Prompting
prompt_zero_shot = "Clasifica la severidad de este reporte clínico en [LEVE, MODERADO, CRÍTICO]: 'Paciente hemodinámicamente inestable con choque séptico en infusión de noradrenalina.'"
print("--- ZERO-SHOT PROMPTING ---")
print(completar_prompt(prompt_zero_shot, temperature=0.0))

print("\n" + "="*70 + "\n")

# Few-Shot Prompting (In-Context Learning)
prompt_few_shot = """Clasifica la severidad del reporte siguiendo los siguientes ejemplos:

Ejemplo 1:
Reporte: Paciente con rinitis alérgica estacional y estornudos ocasionales.
Clasificación: LEVE | Razón: Síntomas ambulatorios sin compromiso sistémico.

Ejemplo 2:
Reporte: Paciente con neumonía adquirida en la comunidad, requiere oxígeno por cánula nasal a 2L/min.
Clasificación: MODERADO | Razón: Requiere soporte oxigenatorio y hospitalización general.

Ejemplo 3:
Reporte: Paciente en paro cardiorrespiratorio con maniobras de RCP avanzada.
Clasificación: CRÍTICO | Razón: Inestabilidad vital inminente.

Ahora clasifica el siguiente reporte:
Reporte: Paciente hemodinámicamente inestable con choque séptico en infusión de noradrenalina.
Clasificación:"""

print("--- FEW-SHOT PROMPTING ---")
print(completar_prompt(prompt_few_shot, temperature=0.0))


## 4. System Prompts, Personas y Asignación de Roles (Diapositiva 08)
El `system_prompt` define las barreras de comportamiento, la audiencia objetivo y el nivel de tecnicismo del modelo.

In [ ]:
pregunta_paciente = "¿Por qué me formularon Metformina si mi glucosa salió en 115 mg/dL?"

# Rol 1: Médico especialista en endocrinología
sys_medico = "Eres un endocrinólogo senior. Explica los conceptos de prediabetes y sensibilidad a la insulina de forma científica y rigurosa."
print("--- ROL 1: ENDOCRINÓLOGO SENIOR ---")
print(completar_prompt(pregunta_paciente, system_prompt=sys_medico, temperature=0.2))

print("\n" + "="*70 + "\n")

# Rol 2: Educador en salud comunitaria
sys_educador = "Eres un educador en salud. Explica la razón del tratamiento con empatía, lenguaje sencillo y sin jerga técnica compleja."
print("--- ROL 2: EDUCADOR EN SALUD ---")
print(completar_prompt(pregunta_paciente, system_prompt=sys_educador, temperature=0.2))


## 5. Salidas Estructuradas con Pydantic (`Structured Outputs`) (Diapositiva 09)
Garantizamos respuestas estrictamente conformes con un esquema de datos definido mediante **Pydantic** y el endpoint `client.beta.chat.completions.parse`.

In [ ]:
# Schema en Pydantic para Triaje Hospitalario
class EvaluacionTriaje(BaseModel):
    paciente_id: str = Field(description="Identificador único del paciente")
    nivel_triage: int = Field(description="Nivel de triaje ESI de 1 (Reanimación) a 5 (No urgente)")
    categoria_color: str = Field(description="Color asignado: Rojo, Naranja, Amarillo, Verde o Azul")
    signos_alarma: List[str] = Field(description="Lista de hallazgos críticos detectados")
    conducta_inmediata: str = Field(description="Acción médica prioritaria a ejecutar")

def evaluar_triaje_con_pydantic(nota_urgencias: str, paciente_id: str = "PAC-2026-88") -> EvaluacionTriaje:
    prompt_user = f"Evalúa la siguiente nota de ingreso a urgencias para el paciente {paciente_id}:\n\n{nota_urgencias}"
    
    if client and hasattr(client, "beta") and hasattr(client.beta.chat.completions, "parse"):
        try:
            completion = client.beta.chat.completions.parse(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "Eres un especialista en triaje de urgencias bajo el protocolo Manchester/ESI."},
                    {"role": "user", "content": prompt_user}
                ],
                response_format=EvaluacionTriaje
            )
            return completion.choices[0].message.parsed
        except Exception as e:
            print(f"[WARN] Error al usar Structured Outputs en OpenAI: {e}")
            
    datos_mock = {
        "paciente_id": paciente_id,
        "nivel_triage": 2,
        "categoria_color": "Naranja",
        "signos_alarma": ["Dolor precordial opresivo", "Diaforesis profiláctica", "SatO2 88%"],
        "conducta_inmediata": "Pasar de inmediato a Sala de Reanimación, monitorización continua y EKG en < 10 min."
    }
    return EvaluacionTriaje.model_validate(datos_mock)

nota_ingreso = "Paciente masculino de 61 años ingresa por dolor torácico retroesternal severo, diaforético, con disnea de reposo y pulsioximetría en 88% al aire ambiente."
resultado_triaje = evaluar_triaje_con_pydantic(nota_ingreso)

print("[OK] Objeto Pydantic `EvaluacionTriaje` recibido y validado:")
print(f"- ID Paciente: {resultado_triaje.paciente_id}")
print(f"- Nivel de Triaje: Nivel {resultado_triaje.nivel_triage} ({resultado_triaje.categoria_color})")
print(f"- Signos de Alarma: {', '.join(resultado_triaje.signos_alarma)}")
print(f"- Conducta Inmediata: {resultado_triaje.conducta_inmediata}")


## 6. Control de Parámetros: Temperature & Top-P (Diapositiva 10)

Los hiperparámetros de generación controlan la distribución de probabilidad del modelo antes de seleccionar el siguiente token:

- **Temperature ($T$)**: Modula la entropía en la función Softmax ($P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$).
  - **$T = 0.0$ (Muestreo Greedy / Determinista)**: El modelo siempre selecciona el token de mayor probabilidad. **Ideal para casos clínicos, dosificación, clasificación JSON y extracción de entidades.**
  - **$T = 0.7 - 1.0$ (Creatividad Moderada)**: Incrementa la diversidad de alternativas. Recomendado para hipótesis diagnósticas diferenciales y muestreos de *Self-Consistency*.
  - **$T > 1.2$ (Alta Entropía)**: Respuestas impredecibles y proclive a alucinaciones.

- **Top-P / Nucleus Sampling ($p$)**: Muestrea dentro del conjunto mínimo de tokens cuya probabilidad acumulada alcanza $p$ (ejemplo: $p = 0.9$). Elimina tokens poco probables.

> **Regla de Oro de OpenAI**: *Ajusta `temperature` O `top_p`, pero NUNCA modifiques ambos al mismo tiempo.*

In [ ]:
def demostrar_efecto_parametros(prompt_test: str):
    print("[DEMOSTRACIÓN DE PARÁMETROS DE INFERENCIA (TEMPERATURE Y TOP-P)]\n")
    
    # 1. Determinista (Temperature = 0.0)
    print("--- 1. Determinista: Temperature = 0.0 (Greedy Decoding) ---")
    res_t0 = completar_prompt(prompt_test, temperature=0.0)
    print(res_t0)
    print("\n" + "="*70 + "\n")
    
    # 2. Creativo (Temperature = 0.9)
    print("--- 2. Exploratorio / Divergente: Temperature = 0.9 ---")
    res_t09 = completar_prompt(prompt_test, temperature=0.9)
    print(res_t09)
    print("\n" + "="*70 + "\n")
    
    # 3. Nucleus Sampling Focalizado (Top-P = 0.1)
    print("--- 3. Nucleus Sampling Focalizado: Top-P = 0.1 (Solo tokens de alta probabilidad) ---")
    res_topp01 = completar_prompt(prompt_test, top_p=0.1)
    print(res_topp01)
    print("\n" + "="*70 + "\n")
    
    # 4. Nucleus Sampling Amplio (Top-P = 0.95)
    print("--- 4. Nucleus Sampling Amplio: Top-P = 0.95 (Mayor diversidad semántica) ---")
    res_topp95 = completar_prompt(prompt_test, top_p=0.95)
    print(res_topp95)

prompt_innovacion = "Menciona 3 líneas de investigación emergentes en el uso de IA Generativa para la detección temprana de enfermedades neurodegenerativas."
demostrar_efecto_parametros(prompt_innovacion)


## 7. Chain-of-Thought (CoT) Reasoning (Diapositiva 11)
Comparamos una solicitud **directa** frente a una estructurada con **Chain of Thought** para resolver un caso multivariable de ajuste de dosificación farmacológica en insuficiencia renal.

In [ ]:
# Caso clínico de dosificación renal
caso_clinico = """
Paciente de 68 años, peso 70 kg, con aclaramiento de creatinina (ClCr) estimado de 35 mL/min.
Se requiere prescribir un antimicrobiano cuya dosis habitual en función renal normal es 500 mg cada 8 horas.
Tabla de ajuste por función renal de la guía clínica:
- ClCr > 50 mL/min: 100% de la dosis habitual (500 mg c/8h -> 1500 mg/día).
- ClCr 30-50 mL/min: Opción A: 500 mg cada 12 horas (1000 mg/día) | Opción B: 250 mg cada 8 horas (750 mg/día).
- ClCr < 30 mL/min: 250 mg cada 12 horas (500 mg/día).
Pregunta: ¿Cuál es el ajuste recomendado y cuál es la dosis total diaria resultante?
"""

print("--- 1. PROMPT ESTÁNDAR DIRECTO (Sin CoT) ---")
prompt_directo = f"Analiza el siguiente caso y da únicamente la dosis e intervalo final recomendados:\n{caso_clinico}"
res_directa = completar_prompt(prompt_directo, temperature=0.0)
print(res_directa)

print("\n" + "="*70 + "\n")

print("--- 2. CHAIN-OF-THOUGHT (Zero-Shot CoT) ---")
prompt_cot = f"""Analiza detalladamente el siguiente caso clínico.
{caso_clinico}

Piensa paso a paso siguiendo esta secuencia de razonamiento:
Paso 1: Identifica el valor exacto de ClCr del paciente.
Paso 2: Compara dicho valor con la tabla de ajuste renal y determina el rango correspondiente.
Paso 3: Analiza las opciones de dosificación disponibles para ese rango.
Paso 4: Calcula la dosis total diaria en mg para cada opción.
Paso 5: Emite la recomendación final justificada.
"""

res_cot = completar_prompt(prompt_cot, temperature=0.0)
print(res_cot)


## 8. Self-Consistency Prompting (Wang et al., 2022) (Diapositiva 12)
La técnica de **Self-Consistency** reemplaza el muestreo 'greedy' (temperatura 0) por la generación de $N$ cadenas de razonamiento independientes a una temperatura moderada (`temperature = 0.7`). Luego, se aplica un **voto por mayoría** sobre la respuesta final extraída de cada ruta.

In [ ]:
def ejecutar_self_consistency(prompt_caso: str, num_muestras: int = 3, temperature: float = 0.7) -> dict:
    """Genera N respuestas independientes con CoT a temperatura > 0 y vota la respuesta final por mayoría."""
    print(f"[SELF-CONSISTENCY] Generando {num_muestras} muestras independientes (Temp={temperature})...\n")
    
    prompt_sc = f"""{prompt_caso}

Razona paso a paso la solución.
Al final de tu respuesta, en una nueva línea, escribe estrictamente el formato:
RESPUESTA_FINAL: [Tu respuesta concisa, ej. 6 mL por toma]
"""
    
    respuestas_completas = []
    respuestas_finales = []
    
    for i in range(num_muestras):
        if client:
            resp = completar_prompt(prompt_sc, temperature=temperature)
        else:
            mock_options = [
                "Paso 1: Dosis diaria total = 18 kg * 50 mg/kg = 900 mg/día.\nPaso 2: Como son 3 tomas, cada toma es 900 / 3 = 300 mg.\nPaso 3: Concentración 250 mg / 5 mL = 50 mg/mL.\nPaso 4: Vol por toma = 300 mg / 50 mg/mL = 6 mL.\nRESPUESTA_FINAL: 6 mL en cada toma (300 mg c/8h)",
                "Paso 1: 18kg x 50mg/kg/día = 900mg total.\nPaso 2: Dividido en 3 tomas = 300mg por toma.\nPaso 3: 250mg en 5mL -> 1mg = 0.02mL -> 300mg = 6 mL.\nRESPUESTA_FINAL: 6 mL en cada toma (300 mg c/8h)",
                "Paso 1: Dosis = 900mg/día. Toma = 300mg.\nPaso 2: Regla de 3: 250mg -> 5mL, 300mg -> x = (300*5)/250 = 6 mL.\nRESPUESTA_FINAL: 6 mL en cada toma (300 mg c/8h)"
            ]
            resp = mock_options[i % len(mock_options)]
            
        respuestas_completas.append(resp)
        
        if "RESPUESTA_FINAL:" in resp:
            ans = resp.split("RESPUESTA_FINAL:")[-1].strip()
        else:
            ans = resp.strip().split("\n")[-1]
            
        respuestas_finales.append(ans)
        print(f"  [Muestra {i+1}]: {ans}\n")
    
    conteo = Counter(respuestas_finales)
    ganador, votos = conteo.most_common(1)[0]
    
    print("="*60)
    print("[VOTO MAYORITARIO (SELF-CONSISTENCY)]:")
    for ans_text, count_v in conteo.items():
        print(f"  * [{count_v}/{num_muestras} votos]: {ans_text}")
    print(f"\n[GANADOR SELECCIONADO]: {ganador}")
    
    return {"ganador": ganador, "votos": votos, "muestras": respuestas_completas}

caso_pediatrico = """
Un paciente pediátrico de 18 kg requiere amoxicilina a dosis de 50 mg/kg/día dividida en 3 tomas iguales diarias.
La presentación disponible en farmacia es suspensión oral de 250 mg / 5 mL.
¿Cuántos mL exactos debe recibir el paciente en CADA TOMA?
"""

resultado_self_consistency = ejecutar_self_consistency(caso_pediatrico, num_muestras=3, temperature=0.7)


## 9. Tree-of-Thoughts (ToT) Prompting (Yao et al., 2023) (Diapositiva 13)
**Tree of Thoughts** amplia el razonamiento mediante la creación de un **árbol de decisión**:
1. **Paso 1 (Branching)**: Generar 3 hipótesis o enfoques estratégicos independientes.
2. **Paso 2 (State Evaluation)**: Evaluar y puntuar la viabilidad y seguridad clínica de cada rama (1 a 10).
3. **Paso 3 (Deepening / Selection)**: Seleccionar la rama con mayor puntuación y desarrollar el plan completo.

In [ ]:
def ejecutar_tree_of_thoughts(problema_complejo: str):
    print("[TREE-OF-THOUGHTS (ToT)] Iniciando exploración en árbol...\n")
    
    prompt_branching = f"""Problema Clínico Complejo:
{problema_complejo}

Propón exactamente 3 enfoques estratégicos o alternativas terapéuticas distintas para abordar esta situación.
Formato:
ENFOQUE A: [Descripción breve]
ENFOQUE B: [Descripción breve]
ENFOQUE C: [Descripción breve]
"""
    print("--- PASO 1: Generación de Ramas (Branching) ---")
    if client:
        ramas_raw = completar_prompt(prompt_branching, temperature=0.7)
    else:
        ramas_raw = """ENFOQUE A: Iniciar cobertura empírica de ultra-amplio espectro con Meropenem + Vancomicina de inmediato.
ENFOQUE B: Solicitar paneles moleculares PCR ultra-rápidos en aspirado traqueal y desescalar según antibiograma previo.
ENFOQUE C: Iniciar Nebulización con Colistina y ajustar dosis sistémica según tasa de filtración glomerular actual."""
    
    print(ramas_raw)
    print("\n" + "="*70 + "\n")
    
    prompt_evaluation = f"""Evalúa los siguientes 3 enfoques clínicos frente al caso: '{problema_complejo}'

Enfoques planteados:
{ramas_raw}

Para cada enfoque, asigna una puntuación de 1 a 10 considerando la nefrotoxicidad y el riesgo de resistencia bacteriana.
Estructura tu respuesta en JSON:
{{
  "evaluaciones": [
    {{"enfoque": "ENFOQUE A", "puntaje": 7, "justificacion": "..."}},
    {{"enfoque": "ENFOQUE B", "puntaje": 9, "justificacion": "..."}},
    {{"enfoque": "ENFOQUE C", "puntaje": 5, "justificacion": "..."}}
  ]
}}
"""
    print("--- PASO 2: Evaluación de Estados (State Evaluation) ---")
    if client:
        eval_raw = completar_prompt(prompt_evaluation, temperature=0.2)
    else:
        eval_raw = json.dumps({
            "evaluaciones": [
                {"enfoque": "ENFOQUE A", "puntaje": 6, "justificacion": "Riesgo elevado de empeorar la falla renal aguda con vancomicina."},
                {"enfoque": "ENFOQUE B", "puntaje": 9, "justificacion": "Maximiza precisión dirigida evitando nefrotoxicidad innecesaria."},
                {"enfoque": "ENFOQUE C", "puntaje": 5, "justificacion": "Uso de colistina presenta alto riesgo de nefrotoxicidad sistémica."}
            ]
        }, indent=2)
    
    print(eval_raw)
    print("\n" + "="*70 + "\n")
    
    prompt_deepening = f"""Teniendo en cuenta las evaluaciones previas:
{eval_raw}

Selecciona la rama ganadora con mayor puntuación y desarrolla el protocolo de manejo clínico detallado paso a paso.
"""
    print("--- PASO 3: Selección de la Rama Óptima y Desarrollo Final ---")
    plan_final = completar_prompt(prompt_deepening, temperature=0.3)
    print(plan_final)

caso_uci = "Paciente de 74 años en UCI con Neumonía Asociada a Ventilador (NAV), Injuria Renal Aguda (AKIN 2) y aislamiento previo de Pseudomonas aeruginosa MDR."
ejecutar_tree_of_thoughts(caso_uci)


## 10. Patrón ReAct & Context Engineering (Scratchpad de Memoria) (Diapositivas 15, 21-23)
Demostración del bucle **ReAct (Reasoning + Acting)** actualizando dinámicamente un **Scratchpad de Memoria** para mantener el estado del contexto entre iteraciones agénticas.

In [ ]:
def herramienta_vademecum_online(farmaco: str) -> str:
    """Simula una herramienta externa de consulta farmacológica."""
    db = {
        "metformina": "Contraindicada si TFG < 30 mL/min por riesgo de acidosis láctica. Reducir 50% dosis si TFG 30-45 mL/min.",
        "glibenclamida": "Contraindicada en insuficiencia renal (TFG < 60 mL/min) por riesgo severo de hipoglucemia prolongada.",
        "empagliflozina": "No recomendada iniciar si TFG < 20 mL/min para control glucémico."
    }
    return db.get(farmaco.lower(), "Fármaco no registrado en la base de datos oficial.")

class AgenteReActConScratchpad:
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt
        self.scratchpad = []
        
    def agregar_evento(self, tipo: str, contenido: str):
        entrada = f"[{tipo.upper()}]: {contenido}"
        self.scratchpad.append(entrada)
        print(f"  |- {entrada}")
        
    def ensamblar_contexto(self, pregunta_usuario: str) -> str:
        memoria = "\n".join(self.scratchpad)
        return f"""<system>
{self.system_prompt}
</system>

<agent_memory_scratchpad>
{memoria}
</agent_memory_scratchpad>

<user_query>
{pregunta_usuario}
</user_query>"""

print("[REACT AGENT] Bucle interactivo con gestión de memoria:\n")
agente = AgenteReActConScratchpad("Eres un agente clínico de apoyo para farmacia hospitalaria.")
consulta = "¿Se puede ajustar Glibenclamida en un paciente con TFG de 40 mL/min o debe suspenderse?"

agente.agregar_evento("thought", "El usuario pregunta por el ajuste de Glibenclamida en TFG de 40 mL/min. Debo consultar la herramienta de vademécum.")
agente.agregar_evento("action", "herramienta_vademecum_online('glibenclamida')")

obs_resultado = herramienta_vademecum_online("glibenclamida")
agente.agregar_evento("observation", obs_resultado)

agente.agregar_evento("thought", "La observación señala que la Glibenclamida está contraindicada si TFG < 60 mL/min debido a hipoglucemia prolongada.")
agente.agregar_evento("final_answer", "Debe suspenderse. La Glibenclamida está contraindicada con TFG < 60 mL/min (el paciente tiene 40 mL/min). Se sugiere rotar a un iSGLT2 o iDPP4 según criterio médico.")

print("\n" + "="*70)
print("[CONTEXTO ENSAMBLADO ENVIADO AL LLM]:")
print(agente.ensamblar_contexto(consulta))
